# (Appendix 1) Chapter 17: Runnables & LCEL

# <font color="#800000">Overview Of Runnables And LCEL</font>

## <font color="#3366FF">Example Of LCEL</font>

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Load OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Create OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define a prompt template
prompt = PromptTemplate.from_template("Explain {topic} in one line")

## Define an output parser to extract text responses
parser = StrOutputParser()

## Create LCEL Chain
chain = prompt | llm | parser

## Invoke Chain
chain.invoke("photosynthesis")


'Photosynthesis is the process by which plants, algae, and some bacteria use sunlight, water, and carbon dioxide to produce glucose (chemical energy) and release oxygen.'

# <font color="#800000">Deep Dive Into Runnabes & LCEL</font>

## <font color="#3366FF">RunnableLambda</font>

### Example

In [3]:
from langchain_core.runnables import RunnableLambda

## Define a transformation function
def to_uppercase(text: str) -> str:
    return text.upper()

## Create custom runnable
custom_runnable = RunnableLambda(to_uppercase)

## Invoke Runnable
custom_runnable.invoke("hello world")


'HELLO WORLD'

In [4]:
from langchain_core.runnables import RunnableLambda

## Create custom runnable using Python Lambda function
custom_runnable = RunnableLambda(lambda x: x.upper())

## Invoke Runnable
custom_runnable.invoke("hello world")


'HELLO WORLD'

#### Async Invoke

In [5]:
## … (same as earlier example)
## Async Invoke Runnable
await custom_runnable.ainvoke("hello world")


'HELLO WORLD'

#### Batch

In [6]:
## … (same as earlier example)
## Batch Runnable
custom_runnable.batch(["hello world","bye world"])


['HELLO WORLD', 'BYE WORLD']

#### Async Batch

In [7]:
## … (same as earlier example)
## Async Batch Runnable
await custom_runnable.abatch(["hello world","bye world"])


['HELLO WORLD', 'BYE WORLD']

#### Batch As Completed

In [8]:
## … (same as earlier example)
## Batch As Completed Runnable
for idx, response in custom_runnable.batch_as_completed(["hello world","bye world"]):
    print(f"Content {idx + 1}: {response}")


Content 1: HELLO WORLD
Content 2: BYE WORLD


#### Async Batch As Completed

In [9]:
## … (same as earlier example)
## Async Batch As Completed Runnable
async for idx, response in custom_runnable.abatch_as_completed(["hello world","bye world"]):
    print(f"Content {idx + 1}: {response}")


Content 1: HELLO WORLD
Content 2: BYE WORLD


#### Stream

In [10]:
import time
from langchain_core.runnables import RunnableLambda

## Define custom function
def to_uppercase_stream(text): 
    for char in text:
        time.sleep(0.25)
        yield str(char).upper()

## Create custom runnable
custom_runnable = RunnableLambda(to_uppercase_stream)

## Stream runnable
for chunk in custom_runnable.stream("Hello World"):
    print(chunk, end="")


HELLO WORLD

#### Async Stream

In [12]:
from langchain_core.runnables import RunnableLambda
import asyncio

## Define custom function (Generator)
async def to_uppercase_stream(text): 
    for char in text:
        await asyncio.sleep(0.25)
        yield str(char).upper()

## Create custom runnable
custom_runnable = RunnableLambda(to_uppercase_stream)

## Async Stream runnable
async for chunk in custom_runnable.astream("Hello World"):
    print(chunk, end="")


HELLO WORLD

#### Async Stream Events

In [13]:
from langchain_core.runnables import RunnableLambda
import asyncio

## Define custom function (Generator)
async def to_uppercase_stream(text): 
    for char in text:
        await asyncio.sleep(0.25)
        yield str(char).upper()

## Create custom runnable
custom_runnable = RunnableLambda(to_uppercase_stream)

## Async Stream runnable
async for event in custom_runnable.astream_events("Hello"):
    print(event, end="")


{'event': 'on_chain_start', 'data': {'input': 'Hello'}, 'name': 'to_uppercase_stream', 'tags': [], 'run_id': '01a13dfa-45c5-72e0-a2cc-909fba5baa5d', 'metadata': {}, 'parent_ids': []}{'event': 'on_chain_stream', 'run_id': '01a13dfa-45c5-72e0-a2cc-909fba5baa5d', 'name': 'to_uppercase_stream', 'tags': [], 'metadata': {}, 'data': {'chunk': 'H'}, 'parent_ids': []}{'event': 'on_chain_stream', 'run_id': '01a13dfa-45c5-72e0-a2cc-909fba5baa5d', 'name': 'to_uppercase_stream', 'tags': [], 'metadata': {}, 'data': {'chunk': 'E'}, 'parent_ids': []}{'event': 'on_chain_stream', 'run_id': '01a13dfa-45c5-72e0-a2cc-909fba5baa5d', 'name': 'to_uppercase_stream', 'tags': [], 'metadata': {}, 'data': {'chunk': 'L'}, 'parent_ids': []}{'event': 'on_chain_stream', 'run_id': '01a13dfa-45c5-72e0-a2cc-909fba5baa5d', 'name': 'to_uppercase_stream', 'tags': [], 'metadata': {}, 'data': {'chunk': 'L'}, 'parent_ids': []}{'event': 'on_chain_stream', 'run_id': '01a13dfa-45c5-72e0-a2cc-909fba5baa5d', 'name': 'to_uppercase_s

### Using RunnableLambda With LCEL Chain

In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

## Load OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Create OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define a prompt template
prompt = PromptTemplate.from_template("Explain {topic} in one sentence.")

## Define an output parser
parser = StrOutputParser()

## Define a transformation function
def to_uppercase(text: str) -> str:
    return text.upper()

## Wrap it with RunnableLambda
uppercase_lambda = RunnableLambda(to_uppercase)

## Create an LCEL chain
chain = prompt | llm | parser | uppercase_lambda

## Invoke the chain
chain.invoke("black holes")


'A BLACK HOLE IS A REGION OF SPACETIME WHERE GRAVITY IS SO STRONG THAT NOTHING, NOT EVEN LIGHT, CAN ESCAPE, DEFINED BY AN EVENT HORIZON THAT ENCLOSES AN EXTREMELY DENSE CORE (A SINGULARITY) OFTEN FORMED WHEN A MASSIVE OBJECT COLLAPSES.'

#### Coercion Of Function In Chain

In [15]:
chain = prompt | llm | parser | to_uppercase

## Invoke the chain
chain.invoke("black holes")


'A BLACK HOLE IS A REGION OF SPACETIME WHERE GRAVITY IS SO INTENSE THAT IT CREATES AN EVENT HORIZON BEYOND WHICH NOTHING—NOT EVEN LIGHT—CAN ESCAPE, TYPICALLY FORMED WHEN A LARGE MASS IS CONCENTRATED INTO A VERY SMALL VOLUME.'

## <font color="#3366FF">RunnablePassthrough</font>

###  Example

In [16]:
from langchain_core.runnables import RunnablePassthrough

## Create an instance of passthrough runnable
passthrough = RunnablePassthrough()

## invoke passthrough runnable
passthrough.invoke("Hello World")


'Hello World'

### Using RunnablePassthrough With LCEL Chain

In [17]:
from langchain_core.runnables import RunnablePassthrough

## Create LCEL chain with only RunnablePassthrough()
chain = RunnablePassthrough() | RunnablePassthrough() | RunnablePassthrough()

## Invoke chain
chain.invoke("Hello World")


'Hello World'

In [18]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

## Create a custom runnable to convert string to uppercase
uppercase_runnable = RunnableLambda(lambda x: x.upper())

## Create a custom runnable to convert string to uppercase
reverse_runnable = RunnableLambda(lambda x: x[::-1])

## Create LCEL chain
chain = uppercase_runnable | RunnablePassthrough() | reverse_runnable

## Invoke chain
chain.invoke("hello world")


'DLROW OLLEH'

## <font color="#3366FF">RunnableSequence</font>

### Example

In [19]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

## Load OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Create OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define a prompt template
prompt = PromptTemplate.from_template("Explain {topic} in one line")

## Define an output parser to extract text responses
parser = StrOutputParser()

## Create LCEL Chain
chain = RunnableSequence(prompt, llm, parser)

## Invoke Chain
chain.invoke("solar eclipse")


"A solar eclipse occurs when the Moon passes between the Sun and Earth, casting its shadow on Earth and partially or completely blocking the Sun's light."

#### Use Of pipe (|) Operator

In [21]:
## Create LCEL Chain with pipe
chain = prompt | llm | parser

## Invoke Chain
chain.invoke("solar eclipse")

"A solar eclipse occurs when the Moon passes between the Sun and Earth, casting a shadow that partially or completely blocks the Sun's light."

#### Batch Processing

In [22]:
## Create LCEL Chain
chain = prompt | llm | parser

## Send Batch Request
chain.batch(["solar eclipse","lunar eclipse"])


['A solar eclipse happens when the Moon passes between the Sun and Earth, blocking all or part of the Sun’s light and casting a shadow on Earth.',
 'A lunar eclipse occurs when the Earth passes between the Sun and the Moon, casting its shadow on and darkening the Moon.']

#### Streaming

In [23]:
## .. (same as earlier example)
## Create LCEL Chain
chain = prompt | llm | parser

## Stream Output
for chunk in chain.stream("molecule"):
    print(chunk,end="", flush=True)


A molecule is the smallest unit of a pure substance made of two or more atoms chemically bonded together that retains the substance's chemical properties.

## <font color="#3366FF">RunnableParallel</font>

### Example

In [24]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

## Define multiple Runnable functions
uppercase_runnable = RunnableLambda(lambda x: x.upper())
reverse_runnable = RunnableLambda(lambda x: x[::-1])
length_runnable = RunnableLambda(lambda x: len(x))

## Create a runnable parallel
runnable_parallel = RunnableParallel(
    
    ## Define output dictionary
    {
        "Uppercase" : uppercase_runnable,
        "Reversed" : reverse_runnable,
        "Length" : length_runnable
    }
)

## Invoke runnable parallel
runnable_parallel.invoke("hello world")


{'Uppercase': 'HELLO WORLD', 'Reversed': 'dlrow olleh', 'Length': 11}

#### Using Alternative Syntax

In [25]:
## Create a runnable parallel
runnable_parallel = RunnableParallel(
    upper=uppercase_runnable,
    reversed=reverse_runnable,
    length=length_runnable
)

## Invoke runnable parallel
runnable_parallel.invoke("hello world")


{'upper': 'HELLO WORLD', 'reversed': 'dlrow olleh', 'length': 11}

## <font color="#3366FF">RunnableBranch</font>

### Example

In [26]:
from langchain_core.runnables import RunnableBranch

## Create Runnable Branch
runnable_branch = RunnableBranch(
(lambda x: "error" in x, RunnableLambda(lambda x: "Handle Error")),
(lambda x: "success" in x, RunnableLambda(lambda x: "Handle Success")),
RunnableLambda(lambda x: "Unknown Input")  # Default case
)

## Print Output of First Input
print(runnable_branch.invoke("error occurred")) 

## Print Output of Second Input
print(runnable_branch.invoke("success case")) 

## Print Output of Third Input
print(runnable_branch.invoke("random text"))


Handle Error
Handle Success
Unknown Input
